Output Parsers for LLM Input / Output with LangChain

PydanticOutputParser
Output Parsers

Output parsers are essential in Langchain for structuring the responses from language models. Below, we will discuss the role of output parsers and include examples using Langchain's specific parser types: PydanticOutputParser, JsonOutputParser, and CommaSeparatedListOutputParser.

Pydantic parser:

This parser allows the specification of an arbitrary Pydantic Model to query LLMs for outputs matching that schema. Pydantic's BaseModel functions similarly to a Python dataclass but includes type checking and coercion.


PydanticOutputParser

This output parser allows users to specify an arbitrary Pydantic Model and query LLMs for outputs that conform to that schema.

Keep in mind that large language models are non-deterministic! You'll have to use an LLM with sufficient capacity to generate well-formed responses.

Use Pydantic to declare your data model. Pydantic's BaseModel is like a Python dataclass, but with actual type checking + coercion

Setup necessary system environment variables

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv()

True

In [3]:
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print("No API key was found!")
elif not api_key.startswith("sk-proj"):
    print("An API key was found, but it doesn't start sk-proj-;")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [4]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

/Users/shardakeshari/Documents/RAGUDEMY/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


                                  PydanticOutputParser

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [6]:
# Define your desired data structure - like a python data class. This will enforce LLM to retrurn with specific datatype.
class QueryResponse(BaseModel):
    description: str = Field(description="A brief description of the topic asked by the user in 100 words")
    pros: str = Field(description="3 bullet points showing the pros of the topic asked by the user")
    cons: str = Field(description="3 bullet points showing the cons of the topic asked by the user")
    conclusion: str = Field(description="One line conclusion of the topic asked by the user")

# Set up a parser + inject instructions into the prompt template.
parser = PydanticOutputParser(pydantic_object = QueryResponse)
parser

PydanticOutputParser(pydantic_object=<class '__main__.QueryResponse'>)

In [7]:
# langchain pre-generated output response formatting instructions, that force LLMs to return structured.
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"description": {"description": "A brief description of the topic asked by the user in 100 words", "title": "Description", "type": "string"}, "pros": {"description": "3 bullet points showing the pros of the topic asked by the user", "title": "Pros", "type": "string"}, "cons": {"description": "3 bullet points showing the cons of the topic asked by the user", "title": "Cons", "type": "string"}, "conclusion": {"description": "One line conclusion of the topic asked by the user", "title": "Conclusion", "type": "string"}}, "required":

In [8]:
# create the final prompt with formatting instructions from the parser
prompt_txt = """
             Asnwer the user query and generated response based on following formatting instruction
             
             Format Instruction:
             {format_instrction}

             Query:
             {query}
             """
            
prompt_txt



'\n             Asnwer the user query and generated response based on following formatting instruction\n\n             Format Instruction:\n             {format_instrction}\n\n             Query:\n             {query}\n             '

In [9]:
prompt = PromptTemplate.from_template(template=prompt_txt)

Difference between PromptTemplate.from_template() and PromptTemplate() is- when using from_template langchain itself identifies placeholders or input varaiabes in prompt. while using PromptTemplate() we have define what all are input variables.

In [10]:
# create a simple LCEL chain to take the prompt, pass it to the LLM, enforce response format using the parser
chain = (prompt
           |
         llm
           |
        parser)

In [11]:
question = "Tell me about Python treat me as beginner"
response = chain.invoke({"query":question, "format_instrction":parser.get_format_instructions()})
response

QueryResponse(description="Python is a high-level, interpreted programming language known for its simplicity and readability. It is widely used for web development, data analysis, artificial intelligence, scientific computing, and more. Python's syntax is designed to be easy to understand, making it an excellent choice for beginners. The language supports multiple programming paradigms, including procedural, object-oriented, and functional programming. With a vast ecosystem of libraries and frameworks, Python allows developers to build applications quickly and efficiently.", pros='- Easy to learn and use, especially for beginners.\n- Extensive libraries and frameworks for various applications.\n- Strong community support and resources available.', cons='- Slower execution speed compared to compiled languages like C or C++.\n- Not ideal for mobile app development.\n- Can consume more memory than some other languages.', conclusion='Overall, Python is an excellent choice for beginners loo

In [12]:
response.dict()

/var/folders/y4/j6_cks6j2fb8p7zcm0hyft6r0000gn/T/ipykernel_43377/3202056457.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  response.dict()


{'description': "Python is a high-level, interpreted programming language known for its simplicity and readability. It is widely used for web development, data analysis, artificial intelligence, scientific computing, and more. Python's syntax is designed to be easy to understand, making it an excellent choice for beginners. The language supports multiple programming paradigms, including procedural, object-oriented, and functional programming. With a vast ecosystem of libraries and frameworks, Python allows developers to build applications quickly and efficiently.",
 'pros': '- Easy to learn and use, especially for beginners.\n- Extensive libraries and frameworks for various applications.\n- Strong community support and resources available.',
 'cons': '- Slower execution speed compared to compiled languages like C or C++.\n- Not ideal for mobile app development.\n- Can consume more memory than some other languages.',
 'conclusion': 'Overall, Python is an excellent choice for beginners l

In [13]:
for k,v in response.dict().items():
    print(f"{k}:\n{v}")

description:
Python is a high-level, interpreted programming language known for its simplicity and readability. It is widely used for web development, data analysis, artificial intelligence, scientific computing, and more. Python's syntax is designed to be easy to understand, making it an excellent choice for beginners. The language supports multiple programming paradigms, including procedural, object-oriented, and functional programming. With a vast ecosystem of libraries and frameworks, Python allows developers to build applications quickly and efficiently.
pros:
- Easy to learn and use, especially for beginners.
- Extensive libraries and frameworks for various applications.
- Strong community support and resources available.
cons:
- Slower execution speed compared to compiled languages like C or C++.
- Not ideal for mobile app development.
- Can consume more memory than some other languages.
conclusion:
Overall, Python is an excellent choice for beginners looking to enter the world 

/var/folders/y4/j6_cks6j2fb8p7zcm0hyft6r0000gn/T/ipykernel_43377/4194237620.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  for k,v in response.dict().items():


JsonOutputParser

This output parser allows users to specify an arbitrary JSON schema and query LLMs for outputs that conform to that schema.

Keep in mind that large language models are non-deterministic! You'll have to use an LLM with sufficient capacity to generate well-formed responses.

It is recommended use Pydantic to declare your data model.

In this example we will use PromptTemplate not PromptTemplate.from_template.

In [14]:
from typing import List
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field


In [15]:
# Define your desired data structure - like a python data class.
class JsonQueryResponse(BaseModel):
    description: str = Field(description="A brief description of the topic asked by the user in 50 words")
    pros: str = Field(description="3 bullet points showing the pros of the topic asked by the user")
    cons: str = Field(description="3 bullet points showing the cons of the topic asked by the user")
    conclusion: str = Field(description="One line conclusion of the topic asked by the user")

# Set up a parser + inject instructions into the prompt template.
parser = JsonOutputParser(pydantic_object = JsonQueryResponse)
parser


JsonOutputParser(pydantic_object=<class '__main__.JsonQueryResponse'>)

In [16]:
# langchain pre-generated output response formatting instructions, that force LLMs to return structured.
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [17]:
# create the final prompt with formatting instructions from the parser
prompt_txt = """
            Answer the query and generate the response based on the following formatted instruction

            Formatted Instruction:
            {format_instrction}

            Query:
            {query}
            """
prompt_txt


'\n            Answer the query and generate the response based on the following formatted instruction\n\n            Formatted Instruction:\n            {format_instrction}\n\n            Query:\n            {query}\n            '

In [18]:
prompt_temp = PromptTemplate(template = prompt_txt, input_variables = ['query'], partial_variables ={"format_instrction":parser.get_format_instructions()})
prompt_temp

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={'format_instrction': 'STRICT OUTPUT FORMAT:\n- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.\n- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).\n- Do not prepend or append any text (e.g., do not write "Here is the JSON:").\n- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output sc

In [19]:
# create a simple LCEL chain to take the prompt, pass it to the LLM, enforce response format using the parser
chain = (prompt_temp
            |
           llm
            |
          parser)
chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={'format_instrction': 'STRICT OUTPUT FORMAT:\n- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.\n- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).\n- Do not prepend or append any text (e.g., do not write "Here is the JSON:").\n- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output sc

In [20]:
topic_queries = [
    "Tell me about commercial real estate",
    "Tell me about Generative AI"
]

topic_queries_formatted = [{"query": topic}
                    for topic in topic_queries]
topic_queries_formatted

[{'query': 'Tell me about commercial real estate'},
 {'query': 'Tell me about Generative AI'}]

In [26]:
responses = chain.map().invoke(topic_queries_formatted)

In [22]:
responses[0]

{'description': 'Commercial real estate refers to properties used exclusively for business purposes, including office buildings, retail spaces, warehouses, and industrial properties. It plays a crucial role in the economy by providing spaces for businesses to operate and grow.',
 'pros': '• Potential for high returns on investment.\n• Long-term leases provide stable income.\n• Diversification of investment portfolio.',
 'cons': '• High initial capital investment required.\n• Market fluctuations can impact property values.\n• Management and maintenance can be complex and time-consuming.',
 'conclusion': 'Commercial real estate can be a lucrative investment but requires careful consideration of risks and management.'}

In [23]:
response.dict().items()

/var/folders/y4/j6_cks6j2fb8p7zcm0hyft6r0000gn/T/ipykernel_43377/4262565446.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  response.dict().items()


dict_items([('description', "Python is a high-level, interpreted programming language known for its simplicity and readability. It is widely used for web development, data analysis, artificial intelligence, scientific computing, and more. Python's syntax is designed to be easy to understand, making it an excellent choice for beginners. The language supports multiple programming paradigms, including procedural, object-oriented, and functional programming. With a vast ecosystem of libraries and frameworks, Python allows developers to build applications quickly and efficiently."), ('pros', '- Easy to learn and use, especially for beginners.\n- Extensive libraries and frameworks for various applications.\n- Strong community support and resources available.'), ('cons', '- Slower execution speed compared to compiled languages like C or C++.\n- Not ideal for mobile app development.\n- Can consume more memory than some other languages.'), ('conclusion', 'Overall, Python is an excellent choice 

In [24]:
for response in responses:
   for k,v in response.items():
    print(f"{k}:\n{v}\n")
    print('-----')

description:
Commercial real estate refers to properties used exclusively for business purposes, including office buildings, retail spaces, warehouses, and industrial properties. It plays a crucial role in the economy by providing spaces for businesses to operate and grow.

-----
pros:
• Potential for high returns on investment.
• Long-term leases provide stable income.
• Diversification of investment portfolio.

-----
cons:
• High initial capital investment required.
• Market fluctuations can impact property values.
• Management and maintenance can be complex and time-consuming.

-----
conclusion:
Commercial real estate can be a lucrative investment but requires careful consideration of risks and management.

-----
description:
Generative AI refers to algorithms that can create new content, such as text, images, and music, by learning from existing data. It has gained popularity due to advancements in machine learning and neural networks, enabling machines to produce human-like output

In [25]:
import pandas as pd

df = pd.DataFrame(responses)
df

,description,pros,cons,conclusion
0,Commercial real estate refers to properties us...,• Potential for high returns on investment.\n•...,• High initial capital investment required.\n•...,Commercial real estate can be a lucrative inve...
1,Generative AI refers to algorithms that can cr...,- Can enhance creativity by providing new idea...,- May produce biased or inappropriate content ...,Generative AI holds great potential but requir...
